## Chronological holdout check

Production evaluation (`kedro run --pipeline modeling`) splits train/val/test at random,
stratified by week. This notebook instead trains on the earliest 80% of weeks and tests
on the most recent 20% — a stricter, realistic check of whether the model actually
generalizes forward in time rather than just interpolating between weeks it's seen
scattered examples of. Everything else (fetch, features, scoring, metrics) calls straight
into `modeling.pipelines.*`, same pattern as `05_inference.ipynb`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from xgboost import XGBRegressor

from modeling.pipelines.raw.nodes import fetch_calls_weekly, fetch_events_weekly, fetch_weather_weekly
from modeling.pipelines.target.nodes import build_target
from modeling.pipelines.features.nodes import (
    drop_incomplete_rows,
    featurize_events,
    featurize_lags,
    featurize_weather,
    join_features,
)
from modeling.pipelines.modeling.nodes import compute_metrics, inference

/home/zaccosenza/code/project-311/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Mirrors conf/base/parameters.yml so this run matches production.
CALLS_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
EVENTS_URL = "https://data.cityofnewyork.us/resource/bkfu-528j.json"
EVENT_INCLUDE_TYPES = [
    "Parade", "Street Festival", "Single Block Festival", "Block Party",
    "Farmers Market", "Street Event", "Religious Event", "Plaza Event",
    "Plaza Partner Event", "Athletic Race / Tour", "Open Street Partner Event",
    "Health Fair", "Sidewalk Sale",
]
WEATHER_LAT, WEATHER_LON = 40.7812, -73.9665
WEATHER_DAILY_VARS = "temperature_2m_max,temperature_2m_min,rain_sum,snowfall_sum"
WEATHER_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_FORECAST_URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"

START_DATE = "2021-01-01"
END_DATE = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")
RAW_DIR = "../data/dev/00_raw"
RAW_FETCH_RETRIES = 3
RAW_FETCH_BACKOFF_SECONDS = 5.0

TARGET_COL = "tgt_calls"
MAX_LAG_WEEKS = 6
YEAR_OFFSET_WEEKS = 52
FEATURE_COLS = [
    "ft_week_of_year",
    "ft_lag_1", "ft_lag_2", "ft_lag_3", "ft_lag_4", "ft_lag_5", "ft_lag_6",
    "ft_lag_1_ly", "ft_lag_2_ly", "ft_lag_3_ly", "ft_lag_4_ly", "ft_lag_5_ly", "ft_lag_6_ly",
    "ft_event_count",
    "ft_lag1_temp_max", "ft_lag1_temp_min", "ft_lag1_had_rain", "ft_lag1_had_snow",
    "ft_pred_temp_max", "ft_pred_temp_min", "ft_pred_had_rain", "ft_pred_had_snow",
    "ft_board_key",
]
CATEGORICAL_FEATURES = ["ft_board_key"]
MODEL_PARAMS = dict(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42, enable_categorical=True)

TRAIN_FRAC = 0.8
RANKING_K = 5

In [3]:
calls = fetch_calls_weekly(
    START_DATE, END_DATE, CALLS_URL, RAW_DIR, RAW_FETCH_RETRIES, RAW_FETCH_BACKOFF_SECONDS,
)
events = fetch_events_weekly(
    START_DATE, END_DATE, EVENTS_URL, EVENT_INCLUDE_TYPES, RAW_DIR, RAW_FETCH_RETRIES, RAW_FETCH_BACKOFF_SECONDS,
)
weather_lag1, weather_pred = fetch_weather_weekly(
    START_DATE, END_DATE, WEATHER_LAT, WEATHER_LON, WEATHER_DAILY_VARS,
    WEATHER_ARCHIVE_URL, WEATHER_FORECAST_URL, RAW_DIR, RAW_FETCH_RETRIES, RAW_FETCH_BACKOFF_SECONDS,
)
print(f"calls: {len(calls):,} rows  |  events: {len(events):,} rows  |  weather: {len(weather_lag1):,} weeks")

calls: 22,467 rows  |  events: 14,542 rows  |  weather: 294 weeks


In [4]:
target = build_target(calls, TARGET_COL)
lag_features = featurize_lags(target, TARGET_COL, MAX_LAG_WEEKS, YEAR_OFFSET_WEEKS)
event_features = featurize_events(events)
weather_features = featurize_weather(weather_lag1, weather_pred)
joined = join_features(target, lag_features, event_features, weather_features, FEATURE_COLS, CATEGORICAL_FEATURES)
data = drop_incomplete_rows(joined, FEATURE_COLS)

print(f"{len(data):,} (board, week) rows with full lag history")
data[["board_key", "week_start"] + FEATURE_COLS].tail()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/15 21:20:29 WARN Utils: Your hostname, ZacPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/15 21:20:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/15 21:20:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


18,408 (board, week) rows with full lag history


,board_key,week_start,ft_week_of_year,ft_lag_1,ft_lag_2,ft_lag_3,ft_lag_4,ft_lag_5,ft_lag_6,ft_lag_1_ly,...,ft_event_count,ft_lag1_temp_max,ft_lag1_temp_min,ft_lag1_had_rain,ft_lag1_had_snow,ft_pred_temp_max,ft_pred_temp_min,ft_pred_had_rain,ft_pred_had_snow,ft_board_key
18403,Unspecified STATEN ISLAND,2026-07-13,29.0,2.833213,3.433987,2.944439,3.433987,3.367296,2.944439,2.995732,...,0.0,29.5,17.8,1.0,0.0,36.8,19.5,1.0,0.0,Unspecified STATEN ISLAND
18404,Unspecified STATEN ISLAND,2026-07-20,30.0,3.433987,2.833213,3.433987,2.944439,3.433987,3.367296,3.367296,...,0.0,35.4,19.6,1.0,0.0,31.7,15.7,1.0,0.0,Unspecified STATEN ISLAND
18405,Unspecified STATEN ISLAND,2026-07-27,31.0,3.401197,3.433987,2.833213,3.433987,2.944439,3.433987,2.944439,...,0.0,29.7,16.8,1.0,0.0,31.2,19.0,1.0,0.0,Unspecified STATEN ISLAND
18406,Unspecified STATEN ISLAND,2026-08-03,32.0,3.295837,3.401197,3.433987,2.833213,3.433987,2.944439,3.135494,...,0.0,29.8,19.3,1.0,0.0,34.2,18.2,1.0,0.0,Unspecified STATEN ISLAND
18407,Unspecified STATEN ISLAND,2026-08-10,33.0,3.218876,3.295837,3.401197,3.433987,2.833213,3.433987,2.890372,...,0.0,33.3,18.4,1.0,0.0,32.1,18.7,1.0,0.0,Unspecified STATEN ISLAND


In [5]:
weeks_sorted = sorted(data["week_start"].unique())
split_idx = int(len(weeks_sorted) * TRAIN_FRAC)
train_weeks, test_weeks = weeks_sorted[:split_idx], weeks_sorted[split_idx:]

data["split"] = np.where(data["week_start"].isin(train_weeks), "train", "test")
train_df = data[data["split"] == "train"]

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
model = XGBRegressor(**MODEL_PARAMS)
model.fit(X_train, np.log1p(y_train))
print(f"train: {len(train_df):,} rows ({train_weeks[0]}..{train_weeks[-1]})  |  test: {(data['split'] == 'test').sum():,} rows ({test_weeks[0]}..{test_weeks[-1]})")

train: 14,664 rows (2022-02-07..2025-09-08)  |  test: 3,744 rows (2025-09-15..2026-08-10)


In [6]:
metrics_df = compute_metrics(model, data, FEATURE_COLS, TARGET_COL, "split", RANKING_K)
metrics_df[metrics_df["split"] == "test"].set_index("metric")["value"]

metric
mae                        120.854643
rmse                       379.311050
baseline_mae               136.998397
model_precision_at_5         0.704167
model_ndcg_at_5              0.948005
model_mrr_at_5               0.693056
baseline_precision_at_5      0.683333
baseline_ndcg_at_5           0.934656
baseline_mrr_at_5            0.621181
Name: value, dtype: float64

In [7]:
test_df = data[data["split"] == "test"]
scored = inference(model, test_df, FEATURE_COLS, TARGET_COL)
pred_col = f"pred_{TARGET_COL}"

hover = [
    f"<b>board {b}</b><br>week: {w}<br>actual: {a:.0f}<br>predicted: {p:.0f}<br>events: {e:.0f}"
    f"<br>pred temp: {tmin:.0f}–{tmax:.0f}°C<br>rain: {rain}  snow: {snow}"
    for b, w, a, p, e, tmin, tmax, rain, snow in zip(
        scored["board_key"], scored["week_start"].astype(str), scored[TARGET_COL], scored[pred_col],
        np.expm1(scored["ft_event_count"]), scored["ft_pred_temp_min"], scored["ft_pred_temp_max"],
        scored["ft_pred_had_rain"], scored["ft_pred_had_snow"],
    )
]

fig = go.Figure()
lims = [0, max(scored[TARGET_COL].max(), scored[pred_col].max())]
fig.add_trace(go.Scatter(x=lims, y=lims, mode="lines", line=dict(color="black", width=1), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(
    x=scored[TARGET_COL], y=scored[pred_col], mode="markers",
    marker=dict(size=6, opacity=0.5, color="#2563eb"), text=hover, hoverinfo="text", showlegend=False,
))
fig.update_layout(
    title="Actual vs predicted — chronological test weeks",
    xaxis_title="Actual calls", yaxis_title="Predicted calls", width=650, height=550,
)
fig.show()

importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values()
fig, ax = plt.subplots(figsize=(6, 8))
ax.barh(importance.index, importance.values)
ax.set_title("Feature importance")
plt.tight_layout()
plt.show()

/tmp/ipykernel_129094/1657980970.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
